In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi


Lamb_shift = -0.01739466142645618 #Lamb shift in GHz
ω_b_0 = 2*ω_a + Lamb_shift*2*pi #7.7*2*pi

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive


ω_d=ω_b_0-Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.9#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

# Check for the Mean Field

In [ ]:
t_max=30/κ_b
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment);

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a[end-5000:end], q_CL_a[end-5000:end], t_list_CL[end-5000:end], ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
t_max=20/κ_b
re_a_0, im_a_0 = 0, 0
a_0 = re_a_0 + 1im* im_a_0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

a, b, t_list_CL = simulator(t_max, system, drive, environment)
a_rot = a.*exp.(1im*ω_a*t_list_CL);

In [ ]:
plot(real.(a_rot), imag.(a_rot))

# Check for the chain

In [ ]:


#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi

ω_b_0 = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0
ω_p=2ω_a-ω_d

ϵ_0 = 0.2#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/baths.jl")
using MPSDynamics

In [ ]:
n=7000

bath=Exp_decay
bath_params =(Ω=50*2*pi, ω_cut_off=20*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ_b, C_inv=C_b_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
chain_params

In [ ]:
t_max=30/κ_b
ϕ_a_0, q_a_0=1.0, 0.0
ϕ_b_0, q_b_0=1.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
environment = OneChain(n, chain_params...)

ϕ_a,ϕ_b,q_a,q_b,t_list = simulator(t_max, system, drive, environment);

In [ ]:
plot(t_list*κ_b,ϕ_b)

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

F,freqs=FFT_perso(ϕ_b, t_list)
f_shifted=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F))
vline!([f_shifted], linestyle=:dash, color=:red, label="f_shifted")
plot!(xlims=(4,8))

In [ ]:
Lamb_shift = f_shifted - ω_b_0/(2*pi)

In [ ]:
t_max=12/κ_b
ϕ_a_0, q_a_0=0.0, 0.0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = OneChain(n, chain_params...)

ϕ_a,ϕ_b,q_a,q_b,t_list = simulator(t_max, system, drive, environment)

In [ ]:
p1,p2 = plot_mode(ϕ_a, q_a, t_list, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

# Effect of removing the Stark shift

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    sin_pump =  - 2*E_J*sin(ϵ_0)
    coupling = (cos((a+conj(a))/sqrt(2) * sqrt(Z_a)/ϕ_quantum + (b+conj(b))/sqrt(2) * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum / sqrt(2)
        
    du[1] = -1im*(ω_a*a + sqrt(Z_a)*sin_pump*coupling) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sqrt(Z_b)*sin_pump*coupling) - p.environment_prepared.κ_b * b
    return nothing
end
#ATS + Mean_Field + ladder_operator
function test_equation2!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    sin_pump =  - 2*E_J*sin(ϵ_0)
    Δ_a = sqrt(Z_a)/(sqrt(2)*ϕ_quantum*ω_a)*sin_pump
    Δ_b = sqrt(Z_b)/(sqrt(2)*ϕ_quantum*ω_b_0)*sin_pump    

    coupling = (cos((a+conj(a)+2*Δ_a)/sqrt(2) * sqrt(Z_a)/ϕ_quantum + (b+conj(b)+2*Δ_b)/sqrt(2) * sqrt(Z_b)/ϕ_quantum)) / ϕ_quantum / sqrt(2)
        
    du[1] = -1im*(ω_a*a + sqrt(Z_a)*sin_pump*coupling) - p.environment_prepared.κ_a * (a + Δ_a)
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sqrt(Z_b)*sin_pump*coupling) - p.environment_prepared.κ_b * (b + Δ_b)
    return nothing
end

In [ ]:
t_max=20/κ_b
sin_pump =  - 2*E_J*sin(ϵ_0)
Δ_a = sqrt(Z_a)/(sqrt(2)*ϕ_quantum*ω_a)*sin_pump
Δ_b = sqrt(Z_b)/(sqrt(2)*ϕ_quantum*ω_b_0)*sin_pump
a_0 = 0
b_0 = 0
a_disp_0 = a_0-Δ_a
b_disp_0 = b_0-Δ_b

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_disp_0, b_disp_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a_disp, b_disp, t_list = simulator(t_max, system, drive, environment, equation = test_equation2!);

a = a_disp.+Δ_a
a_rot = a.*exp.(1im*ω_a*t_list);


plot(real.(a_rot[1:end]), imag.(a_rot[1:end]))

In [ ]:
t_max=100/κ_b
a_0 = 0.0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment, equation = test_equation!);

a_rot = a.*exp.(1im*ω_a*t_list);

plot(real.(a_rot[1:end]), imag.(a_rot[1:end]))